# AutoML-M04: H2O AutoML

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Ejecutar **H2O AutoML** con stacking automático, early stopping y leaderboard
para **Caso D** y **Caso D_strict**.

## 📊 Qué hace H2O AutoML

Framework AutoML distribuido que entrena GBM, XGBoost, DRF, GLM y
StackedEnsemble automáticamente. Usa 5-fold CV interno y genera un
leaderboard ordenado por AUC. Incluye stacking (combinación de modelos).

## 📝 Nota
Datos auditados (F3-M08). Sin leakage, sin constantes, sin traidoras.

## ⚠️ Requisitos
- **Kernel**: `Python (H2O)` — entorno conda `env_h2o`
- **Java 8+** instalado (H2O lo necesita — se auto-detecta)
- Creado con `fautoml_setup_entornos.ipynb` (celda 5)

## 📦 Genera
- `data/automl/results_h2o.parquet` — métricas de todos los modelos
- `docs/html/fase_automl/m04_h2o.html` — documentación visual

## 🔄 Flujo
M03 (PyCaret) → **M04 (H2O)** → M05 (AutoGluon)

## 📌 Siguiente
`fautoml_m05_autogluon.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Qué hace: Verifica/instala dependencias, auto-detecta Java en el sistema,
#   inicia servidor H2O, detecta rutas y verifica datasets limpios.
# Requisitos: Kernel env_h2o + Java 8+ instalado en el sistema.
# ============================================================================

import sys, warnings, time, subprocess, os
from pathlib import Path
warnings.filterwarnings('ignore')

# --- Auto-verificación de dependencias ---
DEPENDENCIAS_REQUERIDAS = {
    'h2o': 'h2o',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'pyarrow': 'pyarrow',
    'jinja2': 'jinja2',
}

faltantes = []
for modulo, paquete_pip in DEPENDENCIAS_REQUERIDAS.items():
    try:
        __import__(modulo)
    except ImportError:
        faltantes.append(paquete_pip)

if faltantes:
    print(f'⚠️ Instalando dependencias faltantes: {faltantes}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + faltantes
    )
    print(f'✅ Instaladas: {faltantes}')
else:
    print('✅ Todas las dependencias disponibles')

# --- Auto-detección de Java ---
# H2O necesita Java 8+ para funcionar. Los entornos conda aislados
# a veces no heredan el PATH del sistema. Esta función busca Java
# en las ubicaciones típicas de cada SO y lo añade al PATH.
def detectar_java():
    """Busca Java en el sistema y lo añade al PATH si no es accesible."""
    # Primero verificar si java ya está en el PATH
    try:
        result = subprocess.run(['java', '-version'], capture_output=True, text=True)
        version_info = result.stderr.strip().split('\n')[0]
        print(f'✅ Java encontrado en PATH: {version_info}')
        return True
    except FileNotFoundError:
        pass

    # Buscar en ubicaciones típicas por SO
    if sys.platform == 'win32':
        # Windows: Eclipse Adoptium/Temurin, Oracle, OpenJDK
        search_roots = [
            r'C:\Program Files\Eclipse Adoptium',
            r'C:\Program Files\Java',
            r'C:\Program Files\Temurin',
            r'C:\Program Files\AdoptOpenJDK',
            r'C:\Program Files\Microsoft\jdk',
            r'C:\Program Files\Zulu',
        ]
    elif sys.platform == 'darwin':
        # macOS: Homebrew, Oracle
        search_roots = [
            '/Library/Java/JavaVirtualMachines',
            '/usr/local/opt/openjdk',
            '/opt/homebrew/opt/openjdk',
        ]
    else:
        # Linux
        search_roots = [
            '/usr/lib/jvm',
            '/usr/java',
            '/opt/java',
        ]

    # JAVA_HOME como opción adicional
    java_home = os.environ.get('JAVA_HOME')
    if java_home:
        bin_path = os.path.join(java_home, 'bin')
        if os.path.exists(bin_path):
            os.environ['PATH'] = bin_path + os.pathsep + os.environ['PATH']
            print(f'✅ Java encontrado via JAVA_HOME: {bin_path}')
            return True

    # Buscar recursivamente en las rutas del SO
    for root_dir in search_roots:
        if not os.path.exists(root_dir):
            continue
        for dirpath, dirnames, filenames in os.walk(root_dir):
            java_exe = 'java.exe' if sys.platform == 'win32' else 'java'
            if java_exe in filenames and os.path.basename(dirpath) == 'bin':
                os.environ['PATH'] = dirpath + os.pathsep + os.environ['PATH']
                # Verificar que funciona
                try:
                    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
                    version_info = result.stderr.strip().split('\n')[0]
                    print(f'✅ Java auto-detectado: {dirpath}')
                    print(f'   {version_info}')
                    return True
                except:
                    continue

    print('❌ Java no encontrado. H2O necesita Java 8+ para funcionar.')
    print('   Instalar desde: https://adoptium.net/ (Eclipse Temurin)')
    return False

if not detectar_java():
    raise RuntimeError('Java es necesario para H2O. Instálalo y reinicia el kernel.')

# --- Detección de entorno (Colab / Local) ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

# --- Imports principales ---
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss)

# --- Verificar H2O e iniciar servidor ---
import h2o
from h2o.automl import H2OAutoML
print(f'✅ H2O {h2o.__version__} importado')

h2o.init(max_mem_size='4G', nthreads=-1)
print('✅ Servidor H2O iniciado')

# --- Imports del proyecto ---
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Rutas y constantes ---
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
RANDOM_STATE = 42
FRAMEWORK = 'H2O'

# DATASETS: Caso D y D_strict (limpios, sin leakage)
DATASETS = {
    'D': RUTA_AUTOML / 'df_exp_automl_D.parquet',
    'D_strict': RUTA_AUTOML / 'dataset_final_tfm.parquet',
}

info_entorno()

# --- Verificación anti-leakage ---
for caso, ruta in DATASETS.items():
    df_tmp = pd.read_parquet(ruta)
    n_cols = df_tmp.shape[1]
    del df_tmp
    print(f'  ✅ Caso {caso}: {ruta.name} ({n_cols} cols) — LIMPIO')
print('✅ Verificación anti-leakage superada')

✅ Todas las dependencias disponibles
✅ Java auto-detectado: C:\Program Files\Eclipse Adoptium\jdk-21.0.10.7-hotspot\bin
   openjdk version "21.0.10" 2026-01-20 LTS
✅ H2O 3.46.0.9 importado
.... not found.r there is an H2O instance running at http://localhost:54321.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-21.0.10+7 (build 21.0.10+7-LTS, mixed mode, sharing)
  Starting server from C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\mjmor\AppData\Local\Temp\tmpwcwaz375
  JVM stdout: C:\Users\mjmor\AppData\Local\Temp\tmpwcwaz375\h2o_mjmor_started_from_python.out
  JVM stderr: C:\Users\mjmor\AppData\Local\Temp\tmpwcwaz375\h2o_mjmor_started_from_python.err
  Server is running at http://127.0.0.1:54321
 successful.o H2O server at http://127.0.0.1:54321 ...


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Europe/Madrid
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,3 months
H2O_cluster_name:,H2O_from_python_mjmor_kftuy1
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.984 Gb
H2O_cluster_total_cores:,22
H2O_cluster_allowed_cores:,22
H2O_cluster_status:,"locked, healthy"


✅ Servidor H2O iniciado
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyec

In [2]:
# ============================================================================
# CELDA 2: EJECUTAR H2O AutoML (CASO D + D_strict)
# ============================================================================
# Qué hace: Para cada caso, carga datos en H2O, ejecuta AutoML con
#   max_runtime_secs=600 (10 min), 5-fold CV, stacking incluido.
#   Luego evalúa cada modelo del leaderboard en test set y recoge
#   métricas en el esquema unificado.
# Genera: DataFrame unificado con todas las métricas por caso.
# ============================================================================

TARGET = 'abandono'
all_results = []

for caso, ruta in DATASETS.items():
    print(f'\n{"="*70}')
    print(f'CASO {caso}: {ruta.name}')
    print(f'{"="*70}')

    # --- Cargar datos ---
    df = pd.read_parquet(ruta)
    X = df.drop(columns=[TARGET])
    n_rows = len(df)
    n_cols = len(df.columns)
    n_feat = len(X.columns)
    print(f"Dataset: {n_rows:,} filas × {n_cols} columnas ({n_feat} features)")
    print(f'Target: {(df[TARGET]==1).sum():,} abandono ({(df[TARGET]==1).mean()*100:.1f}%)')

    # --- Convertir a H2OFrame ---
    print(f'\n📤 Cargando datos en H2O...')
    hf = h2o.H2OFrame(df)
    hf[TARGET] = hf[TARGET].asfactor()  # Clasificación binaria

    # --- Split 70/30 ---
    train, test = hf.split_frame(ratios=[0.7], seed=RANDOM_STATE)
    features = [c for c in hf.columns if c != TARGET]
    print(f'  Train: {train.nrows:,} | Test: {test.nrows:,}')

    # --- Ejecutar H2O AutoML ---
    print(f'\n💧 Ejecutando H2O AutoML...')
    print(f'   ⏱️ Tiempo máximo: 10 minutos...')
    t0 = time.time()

    aml = H2OAutoML(
        max_runtime_secs=600,
        max_models=20,
        seed=RANDOM_STATE,
        sort_metric='AUC',
        nfolds=5,
        verbosity='warn',
        exclude_algos=['DeepLearning']
    )
    aml.train(x=features, y=TARGET, training_frame=train)
    t_total = time.time() - t0

    # --- Leaderboard ---
    lb = aml.leaderboard.as_data_frame()
    print(f'  ✅ {len(lb)} modelos entrenados en {t_total:.1f}s')
    print(f'\n  Leaderboard (top 5):')
    print(f'  {lb.head(5).to_string(index=False)}')

    # --- Evaluar cada modelo en test set ---
    print(f'\n🔄 Evaluando modelos en test set...')
    for i, row_lb in lb.iterrows():
        model_id = row_lb['model_id']
        try:
            modelo = h2o.get_model(model_id)
            perf = modelo.model_performance(test)

            # Nombre limpio del modelo (sin ID largo)
            model_name_clean = model_id.split('_AutoML_')[0]
            # Simplificar: quitar sufijos grid
            if '_grid_' in model_name_clean:
                parts = model_name_clean.split('_grid_')
                model_name_clean = f'{parts[0]}_grid'

            # Extraer métricas del test set
            auc_val = perf.auc() if perf.auc() else 0
            acc_val = perf.accuracy()[0][1] if perf.accuracy() else 0
            prec_val = perf.precision()[0][1] if perf.precision() else 0
            rec_val = perf.recall()[0][1] if perf.recall() else 0
            f1_val = perf.F1()[0][1] if perf.F1() else 0
            mcc_val = perf.mcc()[0][1] if perf.mcc() else 0
            logloss_val = perf.logloss() if perf.logloss() else 999

            all_results.append({
                'caso': caso,
                'framework': FRAMEWORK,
                'model_name': model_name_clean,
                'accuracy': acc_val,
                'balanced_accuracy': 0,
                'precision': prec_val,
                'recall': rec_val,
                'f1': f1_val,
                'auc_roc': auc_val,
                'mcc': mcc_val,
                'kappa': 0,
                'log_loss': round(logloss_val, 4),
                'train_time_sec': round(t_total / len(lb), 2),
            })
        except Exception as e:
            print(f'  ⚠️ Error con {model_id}: {e}')

    print(f'  ✅ {len(lb)} modelos evaluados para caso {caso}')

# --- DataFrame unificado ---
df_resultados = pd.DataFrame(all_results)
df_resultados = df_resultados.sort_values(['caso', 'f1'], ascending=[True, False]).reset_index(drop=True)

# --- Ranking final por caso ---
for caso in DATASETS.keys():
    print(f'\n--- RANKING CASO {caso} (por F1, top 10) ---')
    mask = df_resultados['caso'] == caso
    print(df_resultados.loc[mask, ['model_name', 'accuracy', 'f1', 'auc_roc', 'mcc', 'train_time_sec']].head(10).to_string(index=False))

print(f'\n📊 Total: {len(df_resultados)} resultados')


CASO D: df_exp_automl_D.parquet
Dataset: 33,621 filas × 22 columnas (21 features)
Target: 9,833 abandono (29.2%)

📤 Cargando datos en H2O...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
  Train: 23,645 | Test: 9,976

💧 Ejecutando H2O AutoML...
   ⏱️ Tiempo máximo: 10 minutos...
AutoML progress: |
20:20:53.638: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


  ✅ 22 modelos entrenados en 287.6s

  Leaderboard (top 5):
                                                 model_id      auc  logloss    aucpr  mean_per_class_error     rmse      mse
   StackedEnsemble_AllModels_1_AutoML_1_20260224_202053 0.948050 0.254821 0.905368              0.130488 0.278239 0.077417
StackedEnsemble_BestOfFamily_1_AutoML_1_20260224_202053 0.946258 0.259010 0.902491              0.132099 0.280426 0.078639
           GBM_grid_1_AutoML_1_20260224_202053_model_12 0.946167 0.259685 0.902855              0.137108 0.280688 0.078786
                         GBM_1_AutoML_1_20260224_202053 0.945924 0.259446 0.902574              0.132090 0.281245 0.079099
                         GBM_2_AutoML_1_20260224_202053 0.945454 0.261064 0.902214              0.134320 0.281191 0.079068

🔄 Evaluando modelos en test set...
  ✅ 22 modelos evaluados para caso D

CASO D_strict: dataset_final_tfm.parquet
Dataset: 33,621 filas × 20 columnas (19 features)
Target: 9,833 abandono (29.2%)

📤 C

C:\Users\mjmor\anaconda3\envs\env_h2o\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


  ✅ 22 modelos entrenados en 215.1s

  Leaderboard (top 5):
                                                 model_id      auc  logloss    aucpr  mean_per_class_error     rmse      mse
   StackedEnsemble_AllModels_1_AutoML_2_20260224_202552 0.934137 0.279668 0.890365              0.143158 0.290693 0.084502
StackedEnsemble_BestOfFamily_1_AutoML_2_20260224_202552 0.932444 0.283200 0.887402              0.143413 0.292628 0.085631
           GBM_grid_1_AutoML_2_20260224_202552_model_12 0.932364 0.284014 0.886948              0.145050 0.292866 0.085771
                         GBM_3_AutoML_2_20260224_202552 0.931887 0.284060 0.887324              0.143361 0.292761 0.085709
                         GBM_1_AutoML_2_20260224_202552 0.931841 0.284446 0.886329              0.147242 0.293459 0.086118

🔄 Evaluando modelos en test set...
  ✅ 22 modelos evaluados para caso D_strict

--- RANKING CASO D (por F1, top 10) ---
                    model_name  accuracy       f1  auc_roc      mcc  train_time

In [3]:
# ============================================================================
# CELDA 3: GUARDAR RESULTADOS
# ============================================================================
# Qué hace: Guarda métricas en parquet para la comparativa final (M06).
# Genera: data/automl/results_h2o.parquet
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_h2o.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} filas (caso D + D_strict)')

💾 results_h2o.parquet: 44 filas (caso D + D_strict)


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS Y HTML
# ============================================================================
# Qué hace: Genera gráficos comparativos y documentación HTML.
# Genera: docs/html/fase_automl/m04_h2o.html
# ============================================================================

graficos_b64 = {}
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase_automl', modulo_activo='m04'
)

# --- Gráfico 1: Top 10 modelos por F1 (barras horizontales) ---
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_plot = df_resultados[mask].head(10).copy()
    df_plot = df_plot.sort_values('f1', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    y_pos = np.arange(len(df_plot))

    bars = ax.barh(y_pos, df_plot['f1'], color='#805ad5', alpha=0.85, height=0.6)
    ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=50, zorder=5, label='AUC-ROC')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['model_name'], fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(f'H2O AutoML Caso {caso}: Top 10 Modelos', fontweight='bold', fontsize=14)
    ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1.05)

    for bar, val in zip(bars, df_plot['f1']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8, color='#2d3748')

    plt.tight_layout()
    graficos_b64[f'top10_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Gráfico 2: Top 5 métricas detalladas ---
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_plot = df_resultados[mask].head(5).copy()

    metricas_plot = ['accuracy', 'f1', 'auc_roc', 'mcc']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_plot))
    width = 0.18
    colores = ['#3182ce', '#38a169', '#e53e3e', '#805ad5']

    for j, (met, col) in enumerate(zip(metricas_plot, colores)):
        ax.bar(x + j*width, df_plot[met], width, label=met, color=col)

    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(df_plot['model_name'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'H2O Caso {caso}: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    graficos_b64[f'barras_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Construir HTML ---
contenido_html = ''

contenido_html += generar_seccion_html('Metodología', f'''
<p><strong>H2O AutoML</strong> es un framework distribuido que entrena automáticamente
GBM, XGBoost, DRF, GLM y StackedEnsemble. Incluye stacking (combinación de modelos base)
y optimización de hiperparámetros con grid search.</p>
<div style="background:#faf5ff;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #805ad5;">
  <strong>⚙️ Config:</strong> max_runtime=600s, max_models=20, 5-fold CV, sort=AUC,
  excluye DeepLearning. Java requerido.
</div>
<div style="background:#EBF8FF;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #3182ce;">
  <strong>📌 Evaluación:</strong> Métricas calculadas sobre test set (30%) después del
  entrenamiento con CV.
</div>
''', '🔬')

# Resultados por caso
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_c = df_resultados[mask]
    mejor = df_c.iloc[0]
    n_modelos = len(df_c)

    # Tabla
    tabla = '<table style="width:100%;border-collapse:collapse;">\n'
    tabla += '<tr style="background:#805ad5;color:white;">'
    for col in ['#', 'Modelo', 'Acc', 'Prec', 'Recall', 'F1', 'AUC', 'MCC', 'LogLoss', 'Tiempo']:
        tabla += f'<th style="padding:8px;text-align:center;font-size:11px;">{col}</th>'
    tabla += '</tr>\n'

    for rank, (i, row) in enumerate(df_c.iterrows(), 1):
        bg = '#f7fafc' if rank % 2 == 0 else 'white'
        if rank <= 3:
            medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
            rank_str = medallas[rank]
        else:
            rank_str = str(rank)

        tabla += f'<tr style="background:{bg};">'
        tabla += f'<td style="padding:6px;text-align:center;font-size:11px;">{rank_str}</td>'
        tabla += f'<td style="padding:6px;font-size:11px;">{row["model_name"]}</td>'
        for c in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'mcc']:
            v = row[c]
            color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
            tabla += f'<td style="text-align:center;color:{color};font-size:11px;">{v:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["log_loss"]:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["train_time_sec"]:.1f}s</td></tr>\n'
    tabla += '</table>'

    graf_top10 = f'<img src="data:image/png;base64,{graficos_b64[f"top10_{caso}"]}" style="max-width:100%;border-radius:8px;">'
    graf_barras = f'<img src="data:image/png;base64,{graficos_b64[f"barras_{caso}"]}" style="max-width:100%;border-radius:8px;">'

    desc_caso = 'Alerta temprana (21 features)' if caso == 'D' else 'Producción (19 features)'
    contenido_html += generar_seccion_html(
        f'💧 Caso {caso}: {desc_caso}',
        f'<p><strong>🏆 Mejor:</strong> {mejor["model_name"]} '
        f'(F1={mejor["f1"]:.4f}, AUC={mejor["auc_roc"]:.4f}, MCC={mejor["mcc"]:.4f})</p>'
        f'<p><strong>📊 Modelos evaluados:</strong> {n_modelos}</p>\n'
        f'{graf_top10}\n<br>\n{tabla}\n<br>\n{graf_barras}'
    )

# --- Comparativa acumulada M01-M04 ---
frameworks_prev = []
for fname, flabel in [('results_baselines.parquet', 'Baselines (M01)'),
                       ('results_lazypredict.parquet', 'LazyPredict (M02)'),
                       ('results_pycaret.parquet', 'PyCaret (M03)')]:
    ruta_fw = RUTA_AUTOML / fname
    if ruta_fw.exists():
        frameworks_prev.append((flabel, pd.read_parquet(ruta_fw)))

if frameworks_prev:
    comparativa = '<table style="width:100%;border-collapse:collapse;">\n'
    comparativa += '<tr style="background:#805ad5;color:white;">'
    for col in ['Caso', 'Framework', 'Mejor Modelo', 'F1', 'AUC', 'MCC']:
        comparativa += f'<th style="padding:8px;text-align:center;">{col}</th>'
    comparativa += '</tr>\n'

    for caso in DATASETS.keys():
        for fw_name, df_fw in frameworks_prev:
            mask_fw = df_fw['caso'] == caso
            if 'model_name' in df_fw.columns:
                mask_fw = mask_fw & (~df_fw['model_name'].str.startswith('Dummy'))
            df_fw_caso = df_fw[mask_fw].sort_values('f1', ascending=False)
            if len(df_fw_caso) > 0:
                mejor_fw = df_fw_caso.iloc[0]
                comparativa += f'<tr style="background:#f7fafc;">'
                comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
                comparativa += f'<td style="padding:6px;text-align:center;">{fw_name}</td>'
                comparativa += f'<td style="padding:6px;">{mejor_fw["model_name"]}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["f1"]:.4f}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["auc_roc"]:.4f}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["mcc"]:.4f}</td></tr>\n'

        # H2O (actual)
        mask_h2o = df_resultados['caso'] == caso
        mejor_h2o = df_resultados[mask_h2o].iloc[0]
        comparativa += f'<tr style="background:white;">'
        comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
        comparativa += f'<td style="padding:6px;text-align:center;"><strong>H2O (M04)</strong></td>'
        comparativa += f'<td style="padding:6px;"><strong>{mejor_h2o["model_name"]}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_h2o["f1"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_h2o["auc_roc"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_h2o["mcc"]:.4f}</strong></td></tr>\n'

    comparativa += '</table>'
    contenido_html += generar_seccion_html('🔄 Comparativa: M01 vs M02 vs M03 vs M04', comparativa)

# --- KPIs ---
casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_d = len(df_resultados[df_resultados['caso']==casos_k[0]])

KPIS = [
    {'valor': str(n_modelos_d), 'titulo': 'Modelos'},
    {'valor': f'{mejor_d["f1"]:.4f}', 'titulo': f'Mejor F1 (D)'},
    {'valor': f'{mejor_ds["f1"]:.4f}', 'titulo': f'Mejor F1 (D_strict)'},
    {'valor': '5-fold CV', 'titulo': 'Validación'},
]

# --- Renderizar HTML ---
html = render_base_html(
    titulo='M04: H2O AutoML', icono='💧',
    subtitulo=f'AutoML Distribuido (Caso D + D_strict)',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + contenido_html,
    notebook_nombre='fautoml_m04_h2o.ipynb', notebook_carpeta='fase_automl'
)
ruta_html = RUTA_FASE_AUTOML_HTML / 'm04_h2o.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m04_h2o.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m04_h2o.html


In [5]:
# ============================================================================
# CELDA 5: CERRAR H2O Y RESUMEN
# ============================================================================

# Cerrar servidor H2O (libera memoria y puerto)
h2o.cluster().shutdown()

casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M04 COMPLETADO')
print('='*60)
print(f'Framework: H2O {h2o.__version__}')
print(f'Validación: 5-fold CV + test set')
print(f'Caso D mejor: {mejor_d["model_name"]} (F1={mejor_d["f1"]:.4f}, AUC={mejor_d["auc_roc"]:.4f})')
print(f'Caso D_strict mejor: {mejor_ds["model_name"]} (F1={mejor_ds["f1"]:.4f}, AUC={mejor_ds["auc_roc"]:.4f})')
print(f'Resultados: {ruta_resultados}')
print(f'HTML: {ruta_html}')
print(f'\n🎯 Siguiente: fautoml_m05_autogluon.ipynb')
print('='*60)

H2O session _sid_b16e closed.

✅ AUTOML-M04 COMPLETADO
Framework: H2O 3.46.0.9
Validación: 5-fold CV + test set
Caso D mejor: GBM_4 (F1=0.8095, AUC=0.9453)
Caso D_strict mejor: StackedEnsemble_AllModels_1 (F1=0.7944, AUC=0.9341)
Resultados: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_h2o.parquet
HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m04_h2o.html

🎯 Siguiente: fautoml_m05_autogluon.ipynb
